In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.manifold import TSNE

In [12]:
vocab_w2v = pd.read_csv("../data/output/VOCAB_W2V.csv", sep="|")
vocab_w2v.head()

,term_str,0,1,2,3,4,5,6,7,8,...,90,91,92,93,94,95,96,97,98,99
0,a,-0.076918,0.307668,-0.018858,-0.363742,0.394469,0.300149,0.382173,0.624130,-0.057221,...,-0.008510,0.173102,-0.064713,-0.450167,0.352525,0.344765,0.006750,-0.327284,-0.300511,0.336414
1,aaron,-0.121549,0.237823,0.579024,0.127727,-0.039846,-0.351108,0.078991,0.310827,-0.640818,...,-0.348973,0.087374,-0.453949,-0.355877,0.243234,0.494017,-0.325608,0.308654,-0.599605,-0.067395
2,aaron's,-0.086982,0.074472,0.413678,0.319847,0.112313,-0.028059,-0.041915,0.359314,-0.254140,...,-0.241337,-0.176876,-0.374701,-0.265356,0.374996,0.332725,0.027086,-0.097409,-0.107350,-0.131387
3,abated,0.037336,0.253331,-0.196200,0.042158,0.162734,-0.050281,-0.141764,0.339077,-0.233735,...,-0.035835,0.165954,0.063998,-0.098712,0.295326,0.227306,0.121758,-0.231402,-0.003298,-0.064221
4,abdon,0.036784,0.111374,0.071258,0.107317,0.155672,-0.201936,-0.259829,0.270885,-0.162629,...,-0.033058,0.263291,0.001156,-0.265447,0.216999,0.294079,-0.024512,-0.064198,0.054212,-0.021314


In [13]:
vocab = pd.read_csv("../data/output/VOCAB.csv", sep="|")

# merge frequency info
vocab_w2v = vocab_w2v.merge(
    vocab[["term_str", "n"]],
    on="term_str",
    how="left"
)

# select top 300 most frequent words
top_n = 300
vocab_subset = (
    vocab_w2v
    .sort_values("n", ascending=False)
    .head(top_n)
)

vocab_subset.head()

,term_str,0,1,2,3,4,5,6,7,8,...,91,92,93,94,95,96,97,98,99,n
4702,the,0.035605,0.040434,0.027274,-0.021746,-0.077256,-0.128082,-0.133321,0.029132,-0.375964,...,0.029496,0.038412,-0.230335,0.338272,0.317501,0.046305,-0.161008,0.068075,-0.099534,63924
207,and,-0.022689,0.382257,0.152863,0.036935,0.387037,-0.224837,-0.012673,0.101984,-0.320753,...,-0.049206,0.194076,-0.035731,0.197844,0.352624,0.005144,-0.155631,-0.078616,-0.128614,51696
3239,of,-0.026078,0.207180,0.120260,0.311622,0.009914,-0.013006,-0.203979,0.398119,-0.366617,...,0.280930,0.269644,0.072040,0.469614,0.275573,-0.182417,-0.133054,-0.014417,-0.016485,34617
4800,to,-0.474011,0.152091,-0.054244,0.147676,0.084398,-0.337909,0.521421,0.530209,0.199620,...,0.386533,-0.075019,-0.002710,0.226248,0.208179,0.417413,-0.476496,0.088444,-0.207796,13562
4701,that,0.003657,0.010106,-0.015751,0.086244,0.099124,0.165573,0.537521,0.225721,0.097949,...,0.249090,0.126781,-0.007496,0.552701,0.370042,0.422935,0.147031,0.312052,-0.074813,12913


In [15]:
vocab_w2v.columns

Index(['term_str', '0', '1', '2', '3', '4', '5', '6', '7', '8',
       ...
       '91', '92', '93', '94', '95', '96', '97', '98', '99', 'n'],
      dtype='object', length=102)

In [14]:
embedding_cols = [col for col in vocab_subset.columns 
                  if col not in ["term_str", "n"]]

X = vocab_subset[embedding_cols].values

print("Embedding matrix shape:", X.shape)

Embedding matrix shape: (300, 100)


In [16]:
tsne_model = TSNE(
    n_components=2,
    perplexity=30,
    random_state=5001,
    init="pca",
    learning_rate="auto"
)

X_2d = tsne_model.fit_transform(X)

In [17]:
tsne_df = pd.DataFrame({
    "term_str": terms,
    "x": X_2d[:, 0],
    "y": X_2d[:, 1]
})

In [19]:
import plotly.express as px

fig = px.scatter(
    tsne_df,
    x="x",
    y="y",
    hover_name="term_str",
    title="Word2Vec Embeddings Visualized with t-SNE",
    width=1000,
    height=800
)

fig.update_traces(marker=dict(size=8, opacity=0.7))

fig.update_layout(
    xaxis_title="t-SNE Dimension 1",
    yaxis_title="t-SNE Dimension 2"
)

fig.show()

In [20]:
graph_path = "/Users/nicholasthornton/Downloads/DS 5001/DS-5001-Bible-Analysis-Final-Project/graphs/w2v_tsne_plot.html"

fig.write_html(graph_path)

print("Saved interactive plot to:")
print(graph_path)

Saved interactive plot to:
/Users/nicholasthornton/Downloads/DS 5001/DS-5001-Bible-Analysis-Final-Project/graphs/w2v_tsne_plot.html


A prominent cluster appears in the region where Dimension 1 ranges from approximately -10 to -5 and Dimension 2 ranges from approximately -15 to -10. This cluster includes the words “moses,” “aaron,” “priests,” “priest,” “tabernacle,” “congregation,” “house,” and “altar.” These terms are tightly grouped in the embedding space, indicating that the Word2Vec model learned strong contextual similarity among them.

The cluster reflects the ritual and institutional structure of early Israelite religious life. Moses and Aaron are central leadership figures in the Pentateuch, while priests serve as ritual intermediaries. The tabernacle and altar are sacred locations associated with worship, sacrifice, and covenantal practices. Their spatial proximity in the embedding space suggests that these terms frequently co-occur within similar verse-level contexts, particularly in books such as Exodus, Leviticus, and Numbers.

This cluster demonstrates that the embedding model successfully captured theological and ceremonial relationships within the corpus. Rather than grouping words randomly, the model learned meaningful semantic associations based on shared narrative and ritual contexts.